In [ ]:
import pandas as pd
import joblib

In [ ]:
import psycopg2 as psycopg

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": "rc1b-uh7kdmcx67eomesf.mdb.yandexcloud.net",
    "port": "6432",
    "dbname": "playground_mle_20260614_fb07c65e05",
    "user": "mle_20260614_fb07c65e05",
    "password": "90eff80856bf46ee968ed1e0bdc85990",
}
assert all(
    [var_value != "" for var_value in list(postgres_credentials.values())]
)

connection.update(postgres_credentials)

# определяем название таблицы, в которой хранятся наши данные
TABLE_NAME = "users_churn"


# эта конструкция создаёт контекстное управление для соединения с базой данных
# оператор with гарантирует, что соединение будет корректно закрыто после выполнения всех операций с базой данных
# причём закрыто оно будет даже в случае ошибки при работе с базой данных
# это нужно, чтобы не допустить так называемую "утечку памяти"
with psycopg.connect(**connection) as conn:

    # создаём объект курсора для выполнения запросов к базе данных
    # с помощью метода execute() выполняется SQL-запрос для выборки данных из таблицы TABLE_NAME
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")

        # извлекаем все строки, полученные в результате выполнения запроса
        data = cur.fetchall()

        # получаем список имён столбцов из объекта курсора
        columns = [col[0] for col in cur.description]

# создаём объект DataFrame из полученных данных и имён столбцов
# это позволяет удобно работать с данными в Python с использованием библиотеки Pandas
df = pd.DataFrame(data, columns=columns)

print(f"Размер нашей таблицы: {df.shape[0]} строк; {df.shape[1]} столбцов")

Размер нашей таблицы: 7043 строк; 22 столбцов


Инфиренс модели

In [13]:
# оценка качества модели
def evaluate_model():
    # загрузите результат прошлого шага: fitted_model.pkl
    with open("dvc/models/fitted_model.pkl", "rb") as fd:
        model = joblib.load(fd)

    X_test = pd.read_csv("dvc/split/X_val.csv")
    y_test = pd.read_csv("dvc/split/y_val.csv").squeeze()

    prediction = model.predict(X_test)
    probas = model.predict_proba(X_test)[:, 1]

    from sklearn.metrics import (
        roc_auc_score,
        precision_score,
        recall_score,
        f1_score,
        log_loss,
        confusion_matrix,
    )

    # импортируйте необходимые вам модули

    # заведите словарь со всеми метриками
    metrics = {}

    # посчитайте метрики из модуля sklearn.metrics
    # err_1 — ошибка первого рода
    # err_2 — ошибка второго рода
    _, err1, err2, _ = confusion_matrix(
        y_test, prediction, normalize="all"
    ).ravel()
    auc = roc_auc_score(y_test, probas)
    precision = precision_score(y_test, prediction)
    recall = recall_score(y_test, prediction)
    f1 = f1_score(y_test, prediction)
    logloss = log_loss(y_test, probas)

    # запишите значения метрик в словарь
    metrics["err1"] = err1
    metrics["err2"] = err2
    metrics["auc"] = auc
    metrics["precision"] = precision
    metrics["recall"] = recall
    metrics["f1"] = f1
    metrics["logloss"] = logloss

    print(f"X_test :\n {X_test.head(2).values.tolist()}")
    print(f"y_test: \n {y_test.head(2).values.tolist()}")

    print(f"Predicted values: {prediction[:20]}")
    print(f"Predicted probabilities: {probas[:20]}")
    print(f"True values: {y_test[:20].values}")


if __name__ == "__main__":
    evaluate_model()

X_test :
 [[438, '2014-02-01', 'Two year', 'Yes', 'Credit card (automatic)', 114.05, 8468.2, 'Fiber optic', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Male', 0, 'Yes', 'Yes', 'Yes'], [2280, '2014-02-01', 'Two year', 'Yes', 'Credit card (automatic)', 25.1, 1789.9, 'Fiber optic', 'No', 'No', 'No', 'No', 'No', 'No', 'Male', 1, 'No', 'No', 'Yes']]
y_test: 
 [0, 0]
Predicted values: [0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0]
Predicted probabilities: [0.01870995 0.01606406 0.04094973 0.42260813 0.05680502 0.76684427
 0.1984876  0.2300382  0.00930822 0.6600862  0.40638667 0.01333299
 0.27312199 0.85815072 0.01594397 0.18756193 0.1910203  0.73290991
 0.4276907  0.15088928]
True values: [0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0]


/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.9.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/pytho

In [ ]:
from sklearn.metrics import 
# импортируйте необходимые вам модули

# заведите словарь со всеми метриками
metrics = {}

# посчитайте метрики из модуля sklearn.metrics
# err_1 — ошибка первого рода
# err_2 — ошибка второго рода
_, err1, _, err2 = # ваш код здесь #
auc = # ваш код здесь #
precision = # ваш код здесь #
recall = # ваш код здесь #
f1 = # ваш код здесь #
logloss = # ваш код здесь #

# запишите значения метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [2]:
import os

import mlflow
import joblib
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

with open("dvc/models/fitted_model.pkl", "rb") as fd:
    model = joblib.load(fd)

X_test = pd.read_csv("dvc/split/X_val.csv")
y_test = pd.read_csv("dvc/split/y_val.csv").squeeze()

EXPERIMENT_NAME = "my_model_experiment_4"
RUN_NAME = "run"
REGISTRY_MODEL_NAME = "churn_model_model1"


os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["S3_BUCKET_NAME"] = "s3-student-mle-20260614-fb07c65e05"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri("http://127.0.0.1:5003")


pip_requirements = "requirements.txt"
prediction = model.predict(X_test)
probas = model.predict_proba(X_test)[:, 1]
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {"metadata_model_type": "monthly"}
scripts = [
    "test_model_logging.ipynb",
    "dvc/scripts/data.py",
    "dvc/scripts/evaluate.py",
    "dvc/scripts/fit.py",
]

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    confusion_matrix,
)

metrics = {}

# посчитайте метрики из модуля sklearn.metrics
# err_1 — ошибка первого рода
# err_2 — ошибка второго рода
_, err1, err2, _ = confusion_matrix(
    y_test, prediction, normalize="all"
).ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, probas)

# запишите значения метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    print(f"Input example:\n {input_example}")

    mlflow.log_metrics(metrics)
    mlflow.log_params({"model_type": "monthly", "run_name": RUN_NAME})

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="churn_model_model1",
        registered_model_name=REGISTRY_MODEL_NAME,
        code_paths=scripts,
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata=metadata,
        await_registration_for=60,
        serialization_format="cloudpickle",
    )
    print(f"scripts: {scripts}")

/Users/sergeyuser/Documents/Yandex Практикум/проекты/mlflow-3-logging/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/07/17 16:18:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Input example:
      id  begin_date            type paperless_billing  \
0   438  2014-02-01        Two year               Yes   
1  2280  2014-02-01        Two year               Yes   
2  2235  2018-03-01        Two year                No   
3  4458  2018-04-01  Month-to-month                No   
4  3762  2014-02-01        Two year               Yes   
5  5749  2018-05-01  Month-to-month               Yes   
6  3567  2015-09-01        One year               Yes   
7  2977  2018-07-01  Month-to-month               Yes   
8  5929  2015-01-01        Two year                No   
9  1640  2018-05-01  Month-to-month               Yes   

              payment_method  monthly_charges  total_charges internet_service  \
0    Credit card (automatic)           114.05        8468.20      Fiber optic   
1    Credit card (automatic)            25.10        1789.90      Fiber optic   
2               Mailed check            59.70        1414.20              DSL   
3  Bank transfer (automatic)    

2026/07/17 16:18:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/17 16:18:17 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - dvc-s3 (current: 3.2.0, required: dvc-s3==3.3.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/07/17 16:18:17 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - dvc-s3 (current: 3.2.0, required: dvc-s3==3.3.

scripts: ['test_model_logging.ipynb', 'dvc/scripts/data.py', 'dvc/scripts/evaluate.py', 'dvc/scripts/fit.py']
🏃 View run run at: http://127.0.0.1:5003/#/experiments/11/runs/a7f1664f76d34de38197c60786af8732
🧪 View experiment at: http://127.0.0.1:5003/#/experiments/11


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(len(os.environ["AWS_ACCESS_KEY_ID"]))
print(len(os.environ["AWS_SECRET_ACCESS_KEY"]))

In [ ]:
loaded_model = mlflow.sklearn.load_model(model_uri=model_info.model_uri)
model_predictions = loaded_model.predict(X_test)

assert model_predictions.dtype == int

print(model_predictions[:10])

In [ ]:
import os
import mlflow
from dotenv import load_dotenv

load_dotenv()

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5001

os.environ["MLFLOW_S3_ENDPOINT_URL"] = (
    "https://storage.yandexcloud.net"  # endpoint бакета от Yandex Cloud
)
# os.environ["S3_BUCKET_NAME"] = "s3-student-mle-20260614-fb07c65e05"
# os.environ["AWS_ACCESS_KEY_ID"] = (
#    "??"  # внесите ID ключа бакета, к которому подключён MLflow
# )
# os.environ["AWS_SECRET_ACCESS_KEY"] = (
#    "??"  # внесите ключ бакета, к которому подключён MLflow
# )

mlflow.set_tracking_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)
mlflow.set_registry_uri(
    f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
)

REGISTRY_MODEL_NAME = "churn_model_model1"
client = mlflow.tracking.MlflowClient()
models = client.search_model_versions(
    filter_string=f"name = '{REGISTRY_MODEL_NAME}'"
)

for model in models:
    # print(model)
    print(
        f"Model name: {model.name}, "
        f"Model version: {model.version}, "
        f"run_id: {model.run_id}, status: {model.status}, "
        f"stage: {model.current_stage}"
    )

model_name_1 = models[-1].name
model_version_1 = models[-1].version
model_stage_1 = models[-1].current_stage

model_name_2 = models[-2].name
model_version_2 = models[-2].version
model_stage_2 = models[-2].current_stage

print(f"Текущий stage последней модели: {model_stage_1}")
print(f"Текущий stage предпоследней модели 2: {model_stage_2}")


client.transition_model_version_stage(
    model_name_1, model_version_1, "production"
)
client.transition_model_version_stage(model_name_2, model_version_2, "staging")

client.rename_registered_model(
    name=REGISTRY_MODEL_NAME, new_name=f"{REGISTRY_MODEL_NAME}_b2c"
)

Model name: churn_model_model1, Model version: 11, run_id: fc0d7910177a4f02b1a7b3f2d7c251d0, status: READY, stage: Staging
Model name: churn_model_model1, Model version: 9, run_id: c4a43a316627462b86c4310d0082398a, status: READY, stage: Production
Model name: churn_model_model1, Model version: 8, run_id: b409f85b074c4373968a372a36f314ce, status: READY, stage: Staging
Model name: churn_model_model1, Model version: 7, run_id: 37655adac4f444379a5bb2c13614cc3d, status: READY, stage: Production
Model name: churn_model_model1, Model version: 6, run_id: 91f3ffb28217457e8ba4763644bfbbd8, status: READY, stage: Staging
Model name: churn_model_model1, Model version: 5, run_id: 566b01302def4cf28704349e814a3a30, status: READY, stage: Production
Model name: churn_model_model1, Model version: 4, run_id: e8165a45599f408194425e8569fd30a0, status: READY, stage: Staging
Model name: churn_model_model1, Model version: 3, run_id: d6ca4ce23bdc454d8a2c65a9da8b4c77, status: READY, stage: Production
Model name:

/var/folders/9p/vkzz84_d4_l8l4s8mjml37740000gn/T/ipykernel_2719/1460233020.py:53: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
/var/folders/9p/vkzz84_d4_l8l4s8mjml37740000gn/T/ipykernel_2719/1460233020.py:56: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(model_name_2, model_version_2, "staging")


<ModelVersion: aliases=[], creation_timestamp=1784209725057, current_stage='Staging', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1784282189607, metrics=None, model_id=None, name='churn_model_model1', params=None, run_id='a4872c88230f446c91e0851af8c7cb87', run_link='', source='models:/m-d9ce95652af943b7a532827eb7807632', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>